# Train your own simple Language ID system with Naive Bayes and Character features

This notebook walks through a complete pipeline for **language identification (LID)**: the task of automatically detecting which language a piece of text is written in.

We use a large dataset (~20 million samples, though we also provide code for subsampling the total) from the Mozilla Common Voice corpus, covering over 200 languages.

---

## Datasets

For training, we use the **Mozilla Common Voice** text LID dataset (CV-LID), stored as a single tab-separated file, containing train, dev, and splits. Each row has at minimum two columns: `sentence` (the text) and `lang` (ISO-639-{1,3} language code). The dataset also has a `.csv` histogram of the number of samples per language.


We will evaluate our model both on CV-LID **and** CommonLID, an evaluation-only LID dataset from Common Crawl!



## Imports

We rely entirely on standard scientific Python libraries:

- **pandas** -- for reading TSV files in chunks
- **scikit-learn** -- for the vectorizer and classifier
- **numpy** -- for array operations during evaluation
- **tqdm** -- for a progress bar during the training loop
- **math** -- to estimate the total number of chunks upfront

In [14]:
import math

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report
from tqdm.notebook import tqdm
from fox_progress_bar import ProgressBar

In [2]:
PATH_TO_CV_LID = "mcv_text_lid"
PATH_TO_COMMON_LID = ""

In [31]:
# Create data splits for CV-LID

original_in = open(f"{PATH_TO_CV_LID}/mcv_text_lid.tsv")
train_out = open(f"{PATH_TO_CV_LID}/train.tsv", "w")
dev_out = open(f"{PATH_TO_CV_LID}/dev.tsv", "w")
test_out = open(f"{PATH_TO_CV_LID}/test.tsv", "w")
outfiles = (train_out, dev_out, test_out)

header = next(original_in)
for outfile in outfiles:
    print(header, file=outfile)

# for row in tqdm(original_in, total=20_000_000, desc="Splitting", postfix="🦊"):
print("Splitting all rows into train, dev, and test files...")
pb = ProgressBar(total_size=19762357, unit="rows")
for row in original_in:
    row = row.strip("\n")
    pb.update(1)
    split = row.split("\t")[-1]
    if split == "train":
        print(row, file=train_out)
    elif split == "dev":
        print(row, file=dev_out)
    else:
        print(row, file=test_out)


original_in.close()
for outfile in outfiles:
    outfile.close()

Splitting all rows into train, dev, and test files...
█████████████████████████████████████████████████🦊 100.0% (19769779.0 rows/19762357.0 rows) 706039.9 rows/s ETA: --:--

In [23]:
ls {PATH_TO_CV_LID}/*.tsv

mcv_text_lid/dev.tsv           mcv_text_lid/test.tsv
mcv_text_lid/mcv_text_lid.tsv  mcv_text_lid/train.tsv


## Labels

The cell below defines every language code that appears in the CV-LID dataset. We need this list **before training** because scikit-learn's `partial_fit` requires knowing the full set of possible classes on the very first call.

Language codes follow BCP 47 conventions, using ISO 639-1 two-letter codes for major languages (e.g. fr, de), ISO 639-3 three-letter codes for minority languages (e.g. lua, ewo), and language-region subtags where dialects need to be distinguished (e.g. nb-NO, zh-CN).


In [4]:
with open(f"{PATH_TO_CV_LID}/language_histogram.csv") as f:
    ALL_LANGUAGES = [row.split(",")[0] for row in f][1:]  # ignor header row
    
print(f"Total languages in label set: {len(ALL_LANGUAGES)}")

Total languages in label set: 307


## Feature extraction: HashingVectorizer with character n-grams

### Why character n-grams?

For language ID, **character-level features** consistently outperform word-level features. The key intuitions are:

1. **Script and orthography are highly language-specific.** The sequence `sch` is common in German, `ou` is common in French, etc.
2. **Morphology can be captured implicitly.** Prefixes, suffixes, and inflectional endings are substrings, so character sequence features will capture them.

We use `analyzer='char_wb'` ("character within boundaries"), which pads each token with spaces before extracting n-grams. This means the start and end of words are represented distinctively -- ` he` vs `he ` -- which is useful for identifying language-specific word shapes.

We extract n-grams of length 2, 3, and 4 (`ngram_range=(2, 4)`).

### Why HashingVectorizer?

The standard `CountVectorizer` or `TfidfVectorizer` builds an explicit **vocabulary dictionary** that maps every observed n-gram to a column index. With 20 million sentences and hundreds of languages, that vocabulary can grow to tens of millions of entries -- too large to hold in RAM.

`HashingVectorizer` avoids this entirely by using a **hash function** to map each n-gram directly to a column index in a fixed-size feature matrix. There is no vocabulary object to store, and memory usage is constant regardless of dataset size. The tradeoff is that two different n-grams can hash to the same column (a "collision"), but with `n_features=2**18` (~262,000 buckets) the collision rate is low enough to be harmless in practice.

> **Note:** We set `alternate_sign=False` because `MultinomialNB` requires non-negative feature values. The alternate-sign trick is a variance-reduction technique that introduces negative values, which NB cannot handle.

In [24]:
vectorizer = HashingVectorizer(
    analyzer='char_wb',      # character n-grams padded at word boundaries
    ngram_range=(2, 4),      # bigrams, trigrams, and 4-grams
    n_features=2**18,        # ~262k feature buckets...you can play with this...
    alternate_sign=False,    # keep values non-negative
    norm=None,               # raw counts, not L2-normalized
)


## Multinomial Naive Bayes classification model

### How Naive Bayes works for text

For each language $L$, a Naive Bayes model estimates the probability $P(\text{feature}_i | L)$ -- i.e. how often each n-gram appears in that language. To classify a new text, it computes:

$$\hat{L} = \arg\max_L \; P(L) \cdot \prod_i P(\text{feature}_i | L)^{x_i}$$

where $x_i$ is the count of feature $i$ in the text. In practice this is computed in log-space to avoid numerical underflow.

### Why NB is a good fit here

- **Strong baseline:** Despite the "naive" independence assumption, NB is competitive with more complex models on short-text classification tasks like language ID.
- **Speed:** Fitting NB is just accumulating counts -- extremely fast.
- **Online learning:** `MultinomialNB` supports `partial_fit`, meaning it can update its count tables incrementally from chunks of data. This is essential if we want to train on millions of samples.

### Smoothing

The `alpha` parameter is **Laplace (additive) smoothing**. Without it, any n-gram that never appears in the training data for a given language would give that language a probability of zero -- causing the whole product to collapse to zero. A small `alpha=0.01` avoids this while keeping the estimates close to the empirical frequencies.

In [32]:
clf = MultinomialNB(alpha=0.01)

# --- Alternative: SGDClassifier as a linear SVM surrogate ---
# LinearSVC does NOT support partial_fit. If you want an SVM-style
# decision boundary with online learning, use SGDClassifier with
# loss='hinge'. The interface is identical -- partial_fit works the
# same way. Uncomment to try:
#
# clf = SGDClassifier(
#     loss='hinge',     # hinge loss = linear SVM
#     alpha=1e-5,       # L2 regularization strength
#     max_iter=1,       # one pass per partial_fit call
#     tol=None,         # disable convergence check in online mode
#     n_jobs=-1,        # use all CPU cores
#     random_state=42,
# )

## Processing data: incremental reading/training, or subsampling

### The memory problem

Our 20 million sentences of text is about 1GB in RAM, and the corresponding feature matrix (even sparse) would be enormous. We probably shouldn't call `.fit()` on the whole dataset at once.

### The solutions: `partial_fit` + chunked reading, or subsampling

#### `partial_fit`
Pandas' `read_csv` with `chunksize` returns an **iterator** over DataFrames, each containing at most `CHUNK_SIZE` rows. We vectorize each chunk and call `partial_fit`, which **accumulates** the count statistics rather than restarting from scratch.

Key details:

- **`classes=ALL_LANGUAGES` on first call:** scikit-learn needs to know the full label set upfront to allocate the right number of parameters. After the first call, this argument is ignored.
- **`on_bad_lines='skip':`** Some rows in the TSV may have malformed quoting or extra tab characters. Skipping them is safer than crashing.
- **`dropna`:** `HashingVectorizer` cannot handle `NaN` values. We drop any rows where either the sentence or the label is missing.
- **`sample(frac=1)`:** If the file is sorted by language (common in compiled corpora), the model will see all samples of one language before moving to the next. Shuffling within each chunk partially mitigates this ordering bias.

### Chunk size tuning

- Smaller chunks = lower peak RAM, more Python loop overhead
- Larger chunks = higher RAM, faster wall-clock time
- 500k rows is a reasonable default; push to 1M-2M if your machine has 16+ GB RAM




In [33]:
TRAIN_PATH = f"{PATH_TO_CV_LID}/train.tsv"

In [28]:

TOTAL_ROWS = 20_000_000
CHUNK_SIZE = 500_000


n_chunks = math.ceil(TOTAL_ROWS / CHUNK_SIZE)

reader = pd.read_csv(
    TRAIN_PATH,
    sep="\t",
    chunksize=CHUNK_SIZE,
    on_bad_lines="skip",
)

for i, chunk in enumerate(tqdm(reader, total=n_chunks, desc="Training")):
    # Drop rows with missing text or labels
    chunk = chunk.dropna(subset=["sentence", "lang"])
    
    # Shuffle within chunk to reduce ordering bias
    chunk = chunk.sample(frac=1, random_state=i)
        
    # Transform text to sparse feature matrix
    X = vectorizer.transform(chunk["sentence"])
    y = chunk["lang"]

    # Update model -- pass classes only on the first call
    fit_params = {"classes": ALL_LANGUAGES} if i == 0 else {}
    clf.partial_fit(X, y, **fit_params)

print("Training complete.")

Training:   0%|          | 0/40 [00:00<?, ?it/s]

Training complete.


In [36]:
SAMPLE_RATE = 0.10

chunks = []
for chunk in pd.read_csv("mcv_text_lid/train.tsv", sep="\t", chunksize=500_000, on_bad_lines="skip"):
    chunks.append(chunk.sample(frac=SAMPLE_RATE, random_state=42))

sampled_training_data = pd.concat(chunks)
print(f"Finished subsampling {len(sampled_training_data)} examples from the 20M row file")
X = vectorizer.transform(sampled_training_data["sentence"])
print("Transformed training text into features...")
y = sampled_training_data["lang"]
print("Fitting model...")
clf.fit(X, y)


print("Training complete.")

Finished subsampling 1586135 examples from the 20M row file
Training complete.


## Evaluation on the CV-LID Development Set

We evaluate on `dev.tsv`, a held-out set the model has never seen.

The `classification_report` from scikit-learn gives us, for each language:

| Metric | Meaning |
|---|---|
| **Precision** | Of all texts predicted as language X, what fraction truly are X? |
| **Recall** | Of all texts that truly are language X, what fraction did we correctly identify? |
| **F1** | Harmonic mean of precision and recall |
| **Support** | Number of test samples for this language |

**F1 is usually the most informative single metric** for language ID because the class distribution is highly imbalanced -- some languages have thousands of test sentences, others have fewer than 10.

In [37]:
DEV_PATH = f"{PATH_TO_CV_LID}/dev.tsv"

test = pd.read_csv(DEV_PATH, sep="\t", on_bad_lines="skip")
test = test.dropna(subset=["sentence", "lang"])

X_test = vectorizer.transform(test["sentence"])
y_true = np.array(test["lang"])
y_pred = clf.predict(X_test)

# Build and print the full report
report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
print(classification_report(y_true, y_pred, digits=3, zero_division=0))

              precision    recall  f1-score   support

          ab      1.000     0.999     1.000    104372
         abb      0.973     1.000     0.986       108
         ady      0.890     0.945     0.917       972
          af      0.936     0.975     0.955       477
         ajg      0.717     0.505     0.592       311
         aln      0.842     1.000     0.914       149
          am      0.982     1.000     0.991       224
          an      0.529     0.901     0.667      1054
          ar      0.995     0.997     0.996      5912
          as      0.968     0.992     0.980       773
         ast      0.519     0.437     0.474       158
          az      0.998     0.992     0.995      9410
          ba      0.996     0.985     0.990     15537
         bag      0.947     1.000     0.973       108
         bas      0.937     0.979     0.958       534
         bax      0.943     0.991     0.967       117
         bba      0.969     0.979     0.974        96
         bbj      0.960    

## Evaluating on CommonLID
Now, let's see how well this model performs on the CommonLID evaluation dataset. Since the domains may be quite different, we may not want to 

# TODO

## Diagnostic Tools -- Finding and Understanding Errors

An aggregate F1 score (or Precision or Recall) can hide a lot of important performance variation. Language ID models typically perform **very well on high-resource languages** (English, French, German) and **much worse on low-resource ones** (minority languages with little training data, or languages that are closely related to one another or to higher-resource languages).

The two functions below give us a way to:

1. **Rank all languages by F1** so we can quickly find where the model struggles.
2. **Drill into a specific language** to see which other languages it gets confused with.

Understanding confusions is often more actionable than the aggregate F1. e.g.:

- If `ht` (Haitian Creole) is mostly confused with `fr` (French), that tells us the model needs more contrastive examples of those two languages, or that the n-gram range should be widened to capture longer distinguishing patterns.
- If a minority language is confused with many different languages randomly, it likely just lacks training data.

In [38]:
def show_worst_languages(n=20, min_support=50):
    """
    Print the n languages with the lowest F1 score.

    Parameters
    ----------
    n : int
        How many languages to display.
    min_support : int
        Exclude languages with fewer than this many test samples.
        Languages with very few samples will naturally score near
        zero due to small-sample noise, which can obscure genuinely
        problematic languages that have adequate test coverage.
    """
    scores = {
        lang: report[lang]
        for lang in ALL_LANGUAGES
        if lang in report and report[lang]["support"] >= min_support
    }
    ranked = sorted(scores.items(), key=lambda x: x[1]["f1-score"])[:n]

    print(f"Bottom {n} languages by F1 (min support = {min_support})\n")
    print(f"{'Rank':<6} {'Lang':<14} {'F1':>6}  {'Precision':>10}  {'Recall':>8}  {'Support':>9}")
    print("-" * 60)
    for i, (lang, r) in enumerate(ranked, 1):
        print(
            f"{i:<6} {lang:<14} {r['f1-score']:>6.3f}  "
            f"{r['precision']:>10.3f}  {r['recall']:>8.3f}  "
            f"{int(r['support']):>9}"
        )

In [39]:
def show_confusions(lang, top_n=10):
    """
    For a given ground-truth language, show what the model predicted.

    This is a targeted confusion matrix -- instead of showing the full
    N x N matrix (which would be enormous with 200+ languages), we focus
    on a single row: all test samples where the true label is `lang`.

    Parameters
    ----------
    lang : str
        BCP-47 language code to investigate (e.g. 'ht', 'fr', 'zh-CN').
    top_n : int
        How many predicted labels to show (sorted by frequency).
    """
    mask = y_true == lang
    if mask.sum() == 0:
        print(f"No test samples found for '{lang}'.")
        return

    preds = y_pred[mask]
    total = len(preds)
    correct = (preds == lang).sum()

    print(f"Ground truth: '{lang}'  ({total} samples, {correct / total:.1%} correct)\n")
    print(f"  {'Predicted as':<20} {'Count':>7}  {'Share':>7}")
    print("  " + "-" * 38)

    counts = pd.Series(preds).value_counts()
    for pred_lang, count in counts.head(top_n).items():
        marker = "correct ->" if pred_lang == lang else "          "
        print(f"{marker} {pred_lang:<20} {count:>7}  {count / total:>7.1%}")

### Find the worst-performing languages

Run the cell below to see the 20 languages where the model struggles most. The `min_support=50` filter excludes any language with fewer than 50 test samples, so we are looking at **meaningful failures** rather than statistical noise from tiny test sets.

In [40]:
show_worst_languages(n=20, min_support=50)

Bottom 20 languages by F1 (min support = 50)

Rank   Lang               F1   Precision    Recall    Support
------------------------------------------------------------
1      sco             0.047       0.024     0.985         67
2      qur             0.156       0.213     0.123        106
3      qxa             0.197       0.191     0.204        108
4      qxt             0.199       0.221     0.181         83
5      qva             0.281       0.286     0.277         94
6      qvl             0.300       0.297     0.303        109
7      zh-TW           0.342       0.820     0.216       2061
8      qws             0.350       0.322     0.382        102
9      yue             0.360       0.733     0.238       1864
10     zh-HK           0.399       0.852     0.260       2127
11     qux             0.413       0.432     0.396         96
12     qwa             0.470       0.407     0.556        133
13     ast             0.474       0.519     0.437        158
14     quy             0.

### 7b. Drill into a specific language's confusions

Pick any language from the list above (or any other language code) and call `show_confusions`. The output shows every label the model assigned when the ground truth was that language, sorted by frequency.

**How to interpret this:**

- A single dominant confusion (e.g. 60% of `ht` predicted as `fr`) suggests the two languages are too similar at the n-gram level -- they may need longer n-grams or more training data to pull apart.
- Many small confusions spread across unrelated languages suggests the model is essentially guessing -- likely a data scarcity problem for this language.
- High recall but low precision (lots of correct predictions here, but this language is also over-predicted for others) may indicate that this language's n-gram profile is very common or generic.

In [48]:
# Pass any language code you want to investigate
show_confusions('', top_n=10)

Ground truth: 'ht'  (7 samples, 14.3% correct)

  Predicted as           Count    Share
  --------------------------------------
           fr                         2    28.6%
           rw                         1    14.3%
correct -> ht                         1    14.3%
           en                         1    14.3%
           es                         1    14.3%
           id                         1    14.3%


### Batch diagnosis -- top confusions for the worst performers

This cell automatically finds the 5 worst languages (by F1, with min support) and prints the confusion breakdown for each.

In [49]:
MIN_SUPPORT = 50
N_WORST = 5

scores = {
    lang: report[lang]["f1-score"]
    for lang in ALL_LANGUAGES
    if lang in report and report[lang]["support"] >= MIN_SUPPORT
}
worst_langs = sorted(scores, key=scores.get)[:N_WORST]

for lang in worst_langs:
    print("=" * 50)
    show_confusions(lang, top_n=5)
    print()

Ground truth: 'sco'  (67 samples, 98.5% correct)

  Predicted as           Count    Share
  --------------------------------------
correct -> sco                       66    98.5%
           en                         1     1.5%

Ground truth: 'qur'  (106 samples, 12.3% correct)

  Predicted as           Count    Share
  --------------------------------------
           qxa                       19    17.9%
           qva                       16    15.1%
           qvl                       15    14.2%
correct -> qur                       13    12.3%
           qxt                       13    12.3%

Ground truth: 'qxa'  (108 samples, 20.4% correct)

  Predicted as           Count    Share
  --------------------------------------
           qvl                       25    23.1%
correct -> qxa                       22    20.4%
           qwa                       17    15.7%
           qws                       10     9.3%
           qxt                        8     7.4%

Ground truth: 